<a href="https://colab.research.google.com/github/abdullah-subial/dubai-nlp-engine/blob/main/Dubai_NLP_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fetch Dubai Restaurants Reviews**



In [ ]:
import requests
import pandas as pd
import numpy as np
from google.colab import userdata

# Fetch API key securely from Colab Secrets
API_KEY = userdata.get('GOOGLE_PLACES_API_KEY')

def get_reviews_for_area(area: str, cuisine: str = "", max_budget: float = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fetches restaurant reviews for a specific area in Dubai, dynamically applying cuisine filters,
    calculating empirical price quartiles for missing prices, and filtering by max_budget.

    Returns:
        df_reviews (pd.DataFrame): Extracted review records for NLP analysis.
        df_summary (pd.DataFrame): Data transparency metadata and operational counters.
    """
    url = "https://places.googleapis.com/v1/places:searchText"
    headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": (
        "places.id,"
        "places.displayName,"
        "places.rating,"
        "places.userRatingCount,"     # Total Google ratings
        "places.location,"            # Lat/Lng for Heatmap
        "places.reviews,"             # Includes publishTime & text
        "places.priceLevel,"
        "places.priceRange,"
        "places.primaryTypeDisplayName,"
        "places.formattedAddress,"
        "places.shortFormattedAddress"
    )
}

    # Build query dynamically based on user inputs
    query_string = f"{cuisine} restaurants in {area}, Dubai".strip()
    payload = {"textQuery": query_string, "languageCode": "en", "regionCode": "AE",}

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    places = data.get("places", [])

    # 1. Extract explicit starting prices ONLY if the currency is explicitly AED
    explicit_prices = [
        float(p["priceRange"]["startPrice"]["units"])
        for p in places
        if p.get("priceRange")
        and p["priceRange"].get("startPrice", {}).get("units")
        and p["priceRange"].get("startPrice", {}).get("currencyCode") == "AED" # <-- THE SAFETY CHECK
    ]

    # 2. Compute empirical quartile boundaries for the area
    if len(explicit_prices) >= 4:
        q1, q2, q3 = np.percentile(explicit_prices, [25, 50, 75])
        min_p = min(explicit_prices)
    else:
        # Baseline fallback if explicit price sample size in payload is too small
        min_p, q1, q2, q3 = 25.0, 60.0, 150.0, 350.0

    # 3. Map Google's 4 price categories to the calculated quartiles
    quartile_map = {
        "PRICE_LEVEL_INEXPENSIVE": {"min": min_p, "label": f"AED {int(min_p)} - {int(q1)}"},
        "PRICE_LEVEL_MODERATE": {"min": q1, "label": f"AED {int(q1)} - {int(q2)}"},
        "PRICE_LEVEL_EXPENSIVE": {"min": q2, "label": f"AED {int(q2)} - {int(q3)}"},
        "PRICE_LEVEL_VERY_EXPENSIVE": {"min": q3, "label": f"AED {int(q3)}+"}
    }

    parsed_reviews = []
    exact_price_count = 0
    estimated_price_count = 0
    analyzed_restaurants = set()

    for place in places:
        restaurant_name = place.get("displayName", {}).get("text")
        cuisine_type = place.get("primaryTypeDisplayName", {}).get("text", "General Dining")

        # Extract location coordinates & overall rating counts
        location = place.get("location", {})
        lat = location.get("latitude")
        lng = location.get("longitude")
        user_rating_count = place.get("userRatingCount", 0)

        # Extract actual address
        address = place.get("formattedAddress") or place.get("shortFormattedAddress", "Dubai, UAE")

        # Determine Price & Price Source
        p_range = place.get("priceRange")

        # Check if startPrice exists, has units, AND is strictly in AED
        if (p_range
            and p_range.get("startPrice", {}).get("units")
            and p_range.get("startPrice", {}).get("currencyCode") == "AED"):

            start = float(p_range["startPrice"]["units"])
            end = p_range.get("endPrice", {}).get("units", "")
            min_price = start
            price_display = f"AED {int(start)} - {int(end)}" if end else f"AED {int(start)}+"
            price_source = "Verified Google Price"
            is_exact = True

        else:
            # Fallback for missing prices or non-AED currencies (USD, EUR, etc.)
            raw_level = place.get("priceLevel")
            meta = quartile_map.get(raw_level, {"min": 0, "label": "N/A"})
            min_price = meta["min"]
            price_display = meta["label"]
            price_source = "Area Quartile Estimate"
            is_exact = False

        # Filter out venues exceeding user's maximum budget
        if max_budget is not None and min_price > max_budget:
            continue

        # Track metadata counters
        analyzed_restaurants.add(restaurant_name)
        if is_exact:
            exact_price_count += 1
        else:
            estimated_price_count += 1

        # Extract Reviews with timestamps
        for review in place.get("reviews", []):
            text = review.get("text", {}).get("text")
            publish_time = review.get("publishTime", "")

            if text:
                parsed_reviews.append({
                    "restaurant_name": restaurant_name,
                    "cuisine": cuisine_type,
                    "price_range": price_display,
                    "price_source": price_source,
                    "review_rating": review.get("rating"),
                    "review_text": text,
                    "publish_time": publish_time,           # For Recent Top 3 reviews & date filter
                    "user_rating_count": user_rating_count, # For total review volume column
                    "latitude": lat,                        # For spatial heatmap mapping
                    "longitude": lng,                        # For spatial heatmap mapping
                    "formatted_address": address
                })

    # Construct detailed reviews DataFrame
    df_reviews = pd.DataFrame(parsed_reviews)

    # Construct transparency summary DataFrame
    df_summary = pd.DataFrame([{
        "total_restaurants": len(analyzed_restaurants),
        "total_reviews": len(df_reviews),
        "exact_price_count": exact_price_count,
        "estimated_price_count": estimated_price_count,
        "transparency_note": f"Analyzed {len(analyzed_restaurants)} venues across {len(df_reviews)} reviews. "
                             f"{exact_price_count} using direct menu prices, "
                             f"{estimated_price_count} estimated via local area quartiles."
    }])

    return df_reviews, df_summary


# ==========================================
# EXECUTION & TEST BLOCK
# ==========================================

# Define search parameters
target_area = "Dubai Marina"
target_cuisine = "Italian"
target_max_budget = 150.0

# Run function & unpack both DataFrames
df_reviews, df_summary = get_reviews_for_area(
    area=target_area,
    cuisine=target_cuisine,
    max_budget=target_max_budget
)

# Display transparency banner
print("--- TRANSPARENCY SUMMARY ---")
print(df_summary.loc[0, "transparency_note"])

# Preview reviews DataFrame
print("\n--- REVIEWS DATAFRAME PREVIEW ---")
df_reviews.head(20)

--- TRANSPARENCY SUMMARY ---
Analyzed 19 venues across 95 reviews. 18 using direct menu prices, 1 estimated via local area quartiles.

--- REVIEWS DATAFRAME PREVIEW ---


,restaurant_name,cuisine,price_range,price_source,review_rating,review_text,publish_time,user_rating_count,latitude,longitude,formatted_address
0,Oliva Italian Restaurant,Italian restaurant,AED 50 - 150,Verified Google Price,5,Yet again amazing service. Joanna was great th...,2026-06-16T21:17:34.787077789Z,1912,25.079209,55.140964,Orra Harbour Tower - Shop 1 - Dubai Marina - D...
1,Oliva Italian Restaurant,Italian restaurant,AED 50 - 150,Verified Google Price,5,We love this restaurant and always try to visi...,2026-07-26T12:46:34.086115047Z,1912,25.079209,55.140964,Orra Harbour Tower - Shop 1 - Dubai Marina - D...
2,Oliva Italian Restaurant,Italian restaurant,AED 50 - 150,Verified Google Price,4,"Visited Oliva Cafe recently. Our server, Danni...",2026-05-11T10:13:38.857194630Z,1912,25.079209,55.140964,Orra Harbour Tower - Shop 1 - Dubai Marina - D...
3,Oliva Italian Restaurant,Italian restaurant,AED 50 - 150,Verified Google Price,5,"We visited oliva restaurant today, and it was ...",2026-06-24T14:43:02.094010008Z,1912,25.079209,55.140964,Orra Harbour Tower - Shop 1 - Dubai Marina - D...
4,Oliva Italian Restaurant,Italian restaurant,AED 50 - 150,Verified Google Price,5,Nice Italian restaurant just outside marina . ...,2026-01-01T13:53:10.223081522Z,1912,25.079209,55.140964,Orra Harbour Tower - Shop 1 - Dubai Marina - D...
5,Massimo’s Italian Restaurant - Dubai Marina,Italian restaurant,AED 50 - 250,Verified Google Price,5,We had a lovely dinner. 💕\n\nThe food was simp...,2026-03-26T10:45:12.738708236Z,2955,25.082460,55.142528,Park Island - Dubai Marina- IMPORTANT FOR TAXI...
6,Massimo’s Italian Restaurant - Dubai Marina,Italian restaurant,AED 50 - 250,Verified Google Price,4,"Massimo, Dubai, UAE.\nThe pizza was very good;...",2026-05-22T01:39:50.643747708Z,2955,25.082460,55.142528,Park Island - Dubai Marina- IMPORTANT FOR TAXI...
7,Massimo’s Italian Restaurant - Dubai Marina,Italian restaurant,AED 50 - 250,Verified Google Price,5,I had a wonderful experience at Massimo in Dub...,2026-03-14T17:26:36.838820342Z,2955,25.082460,55.142528,Park Island - Dubai Marina- IMPORTANT FOR TAXI...
8,Massimo’s Italian Restaurant - Dubai Marina,Italian restaurant,AED 50 - 250,Verified Google Price,5,"Delicious 😋I ordered a turkey ham pizza, a gar...",2026-06-19T22:25:13.967461088Z,2955,25.082460,55.142528,Park Island - Dubai Marina- IMPORTANT FOR TAXI...
9,Massimo’s Italian Restaurant - Dubai Marina,Italian restaurant,AED 50 - 250,Verified Google Price,5,Amazing spot and perfect Valentine's dinner\nW...,2026-02-14T15:46:11.569547355Z,2955,25.082460,55.142528,Park Island - Dubai Marina- IMPORTANT FOR TAXI...


In [ ]:
import numpy as np
import pandas as pd

def compute_advanced_metrics(df_reviews: pd.DataFrame) -> pd.DataFrame:
    """
    Computes all advanced metrics and the final composite score for each restaurant.
    Returns a venue-level DataFrame sorted by highest model_score.
    """
    # 1. Take top 5 latest reviews per restaurant
    df_reviews["publish_time"] = pd.to_datetime(df_reviews["publish_time"], errors="coerce")
    top5_reviews = (
        df_reviews.sort_values(by=["restaurant_name", "publish_time"], ascending=[True, False])
        .groupby("restaurant_name")
        .head(5)
    )

    # 2. Extract static venue metadata (1 row per restaurant)
    venues = top5_reviews.groupby("restaurant_name", as_index=False).first()
    venues = venues.rename(columns={"user_rating_count": "total_google_ratings"})

    # 3. Recalculate true averages across those 5 reviews
    venues["avg_google_rating"] = venues["restaurant_name"].map(
        top5_reviews.groupby("restaurant_name")["review_rating"].mean()
    ).round(1)

    if "sentiment_label" in top5_reviews.columns:
        venues["positive_sentiment_pct"] = venues["restaurant_name"].map(
            top5_reviews.groupby("restaurant_name")["sentiment_label"].apply(
                lambda s: (s == "POSITIVE").mean() * 100.0
            )
        ).round(1)
    else:
        venues["positive_sentiment_pct"] = venues["restaurant_name"].map(
            top5_reviews.groupby("restaurant_name")["review_rating"].apply(
                lambda r: (r >= 4).mean() * 100.0
            )
        ).round(1)

    # Clean display fields & price conversions
    venues["short_formatted_address"] = venues["formatted_address"].apply(
        lambda addr: str(addr).split("-")[0].strip()
    )
    venues["price_numeric"] = venues["price_range"].apply(
        lambda x: int(str(x).split("-")[0].replace("AED", "").strip()) if pd.notnull(x) and "-" in str(x) else 150
    )

    # -------------------------------------------------------------------------
    # CALCULATE THE 5 METRICS
    # -------------------------------------------------------------------------
    # 1. Volume Confidence
    venues["volume_confidence_score"] = venues["total_google_ratings"].apply(
        lambda x: min(100.0, (np.log10(x + 1) / np.log10(2500.0)) * 100.0)
    )

    # 2. Sentiment Momentum
    venues["avg_google_rating_scaled"] = (venues["avg_google_rating"] / 5.0) * 100.0
    venues["sentiment_momentum_score"] = np.clip(
        50.0 + (venues["positive_sentiment_pct"] - venues["avg_google_rating_scaled"]),
        0.0, 100.0
    )

    # 3. Aspect Sentiment (Food & Service)
    food_kw = {"food", "dish", "taste", "delicious", "flavor", "menu", "portion", "cooked", "quality", "pizza", "pasta"}
    service_kw = {"staff", "service", "waiter", "waitress", "manager", "attentive", "friendly", "slow", "rude"}

    aspect_scores = {}
    for name, group in top5_reviews.groupby("restaurant_name"):
        f_pos, f_tot, s_pos, s_tot = 0, 0, 0, 0
        for _, row in group.iterrows():
            text = str(row.get("review_text", "")).lower()
            is_pos = (row.get("sentiment_label") == "POSITIVE") if "sentiment_label" in top5_reviews.columns else (row.get("review_rating", 5) >= 4)
            if any(k in text for k in food_kw):
                f_tot += 1
                if is_pos: f_pos += 1
            if any(k in text for k in service_kw):
                s_tot += 1
                if is_pos: s_pos += 1

        f_pct = (f_pos / f_tot * 100.0) if f_tot > 0 else None
        s_pct = (s_pos / s_tot * 100.0) if s_tot > 0 else None

        if f_pct is not None and s_pct is not None:
            aspect_scores[name] = (f_pct + s_pct) / 2.0
        elif f_pct is not None:
            aspect_scores[name] = f_pct
        elif s_pct is not None:
            aspect_scores[name] = s_pct
        else:
            aspect_scores[name] = venues.loc[venues["restaurant_name"] == name, "positive_sentiment_pct"].values[0]

    venues["aspect_sentiment_score"] = venues["restaurant_name"].map(aspect_scores).fillna(50.0)

    # 4. Negativity Risk Penalty
    risk_scores = {}
    for name, group in top5_reviews.groupby("restaurant_name"):
        high_risk = group.apply(
            lambda r: (r.get("review_rating", 5) <= 2) or (r.get("sentiment_label") == "NEGATIVE" and r.get("sentiment_score", 0.0) >= 0.85),
            axis=1
        )
        risk_scores[name] = float((high_risk.mean()) * 100.0)

    venues["negativity_risk_score"] = venues["restaurant_name"].map(risk_scores).fillna(0.0)

    # 5. Price-to-Value Index
    pos_val_terms = {"worth it", "value for money", "reasonable", "fair price", "generous"}
    neg_val_terms = {"overpriced", "rip off", "expensive for what it is", "too small", "not worth"}

    value_scores = {}
    for name, group in top5_reviews.groupby("restaurant_name"):
        pos_c, neg_c = 0, 0
        for text in group["review_text"].astype(str):
            t = text.lower()
            pos_c += sum(1 for term in pos_val_terms if term in t)
            neg_c += sum(1 for term in neg_val_terms if term in t)
        tot = pos_c + neg_c
        value_scores[name] = (pos_c / tot * 100.0) if tot > 0 else 50.0

    venues["value_keyword_score"] = venues["restaurant_name"].map(value_scores).fillna(50.0)
    venues["price_factor"] = venues["price_numeric"].apply(lambda p: max(50.0, 100.0 - ((p / 500.0) * 50.0)))
    venues["price_value_score"] = (
        (0.50 * venues["positive_sentiment_pct"]) +
        (0.30 * venues["value_keyword_score"]) +
        (0.20 * venues["price_factor"])
    )

    # -------------------------------------------------------------------------
    # COMPOSITE SCORE
    # -------------------------------------------------------------------------
    raw_model_score = (
        (0.30 * venues["positive_sentiment_pct"]) +
        (0.25 * venues["avg_google_rating_scaled"]) +
        (0.15 * venues["volume_confidence_score"]) +
        (0.10 * venues["sentiment_momentum_score"]) +
        (0.10 * venues["aspect_sentiment_score"]) +
        (0.10 * venues["price_value_score"]) -
        (0.10 * venues["negativity_risk_score"])
    )

    tie_breaker = (venues["total_google_ratings"] % 97) * 0.001
    venues["model_score"] = np.clip(raw_model_score + tie_breaker, 0.0, 100.0).round(2)

    # Return the fully scored dataframe sorted by rank
    df_scored = venues.sort_values(by="model_score", ascending=False).reset_index(drop=True)
    return df_scored

In [ ]:
df_venues = compute_advanced_metrics(df_reviews)

df_reviews_with_scores = df_reviews.merge(
    df_venues[["restaurant_name", "model_score"]],
    on="restaurant_name",
    how="left"
)
df_reviews_with_scores.head()

,restaurant_name,cuisine,price_range,price_source,review_rating,review_text,publish_time,total_google_ratings,latitude,longitude,...,volume_confidence_score,avg_google_rating_scaled,sentiment_momentum_score,aspect_sentiment_score,negativity_risk_score,value_keyword_score,price_factor,price_value_score,model_score_x,model_score_y
0,Carluccio’s – Dubai Marina Mall,Italian restaurant,AED 50 - 150,Verified Google Price,5,Carluccio’s is one of the best Italian restaur...,2026-07-25 10:30:48.305495308+00:00,8070,25.076766,55.139439,...,100.000000,100.0,50.0,100.0,0.0,100.0,95.0,99.0,94.92,93.42
1,PASTAMAMMA,Restaurant,AED 100 - 300,Verified Google Price,5,"The combination of food, service and atmospher...",2026-08-04 19:48:20.209444300+00:00,3466,25.078184,55.124041,...,100.000000,100.0,50.0,100.0,0.0,100.0,90.0,98.0,94.87,93.37
2,Bussola,Italian restaurant,AED 150 - 400,Verified Google Price,5,Simply gorgeous! Consistently high food qualit...,2026-07-14 15:20:21.126813341+00:00,2533,25.093885,55.148264,...,100.000000,100.0,50.0,100.0,0.0,50.0,85.0,82.0,93.21,93.21
3,ILOLI - Restaurant in Dubai Marina Walk,Restaurant,AED 50 - 300,Verified Google Price,5,One of the best places i visited in dubai the ...,2026-01-23 19:40:03.779004209+00:00,1210,25.085725,55.144764,...,90.735685,96.0,54.0,100.0,0.0,100.0,95.0,99.0,92.96,92.06
4,Massimo’s Italian Restaurant - Dubai Marina,Italian restaurant,AED 50 - 250,Verified Google Price,5,"Delicious 😋I ordered a turkey ham pizza, a gar...",2026-06-19 22:25:13.967461088+00:00,2955,25.082460,55.142528,...,100.000000,96.0,54.0,100.0,0.0,50.0,95.0,84.0,92.85,93.44


**Run Hugging Face Sentiment Analysis Model**

In [ ]:
from collections import Counter
import pandas as pd
import spacy
from spacy.cli import download
from transformers import pipeline

# --- 1. SAFE SPACY MODEL LOADER ---
def load_spacy_model():
    """Safely loads spaCy model; downloads automatically if missing locally or in Colab."""
    try:
        return spacy.load("en_core_web_sm")
    except OSError:
        download("en_core_web_sm")
        return spacy.load("en_core_web_sm")

nlp = load_spacy_model()

# --- 2. GLOBAL TRANSFORMER PIPELINE ---
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0  # Uses GPU in Colab
)

# --- 3. DYNAMIC DISH & VIBE EXTRACTION ---
def extract_dish_and_vibe(texts: list) -> tuple[str, str]:
    """
    Dynamically extracts dish candidates and vibe keywords using spaCy POS tagging:
    - Vibes: Extracts 8-10 top adjectives for rich WordCloud visuals.
    - Famous Dish: Strict noun-chunk filtering to exclude pronouns ('I', 'We', 'It') and non-food terms.
    """
    if not texts:
        return "Chef Special", "Welcoming, Cozy, Lively, Warm, Friendly, Excellent, Great, Elegant"

    doc = nlp(" ".join(texts))

    # 1. Vibe Check: Extract 8-10 top Adjectives for WordCloud
    vibes = [
        token.lemma_.title() for token in doc
        if token.pos_ == "ADJ"
        and not token.is_stop
        and token.is_alpha
        and len(token.text) > 2
    ]

    top_vibes = [v[0] for v in Counter(vibes).most_common(10)]
    vibe_check = ", ".join(top_vibes) if top_vibes else "Welcoming, Cozy, Lively, Warm, Friendly, Excellent, Great, Elegant"

    # 2. Famous Dish: Strict Pronoun & Generic Non-Food Filtering
    non_dish_words = {
        "i", "we", "it", "they", "you", "he", "she", "me", "us", "them",
        "my", "our", "their", "your", "his", "her", "its", "everything", "something",
        "place", "service", "food", "restaurant", "experience", "staff", "table",
        "time", "portion", "spot", "menu", "night", "dinner", "lunch", "visit",
        "quality", "thing", "ambiance", "atmosphere", "price", "value", "view", "way"
    }

    dish_candidates = []
    for chunk in doc.noun_chunks:
        # Filter out chunks whose main root is a pronoun (e.g. "I", "We", "It")
        if chunk.root.pos_ == "PRON":
            continue

        # Keep only core Nouns/Proper Nouns, ignoring stop words & articles
        core_words = [
            token.text.lower() for token in chunk
            if token.pos_ in ("NOUN", "PROPN")
            and not token.is_stop
            and token.is_alpha
        ]

        if not core_words:
            continue

        candidate = " ".join(core_words).title()
        candidate_words = set(candidate.lower().split())

        # Ensure no word in candidate is a pronoun or generic restaurant term
        if not candidate_words.intersection(non_dish_words) and len(candidate.split()) <= 3:
            dish_candidates.append(candidate)

    top_dishes = Counter(dish_candidates).most_common(1)
    famous_dish = top_dishes[0][0] if top_dishes else "Chef Special"

    return famous_dish, vibe_check

# --- 4. SENTIMENT INFERENCE ENGINE ---
def analyze_sentiment(df_reviews: pd.DataFrame) -> pd.DataFrame:
    """
    Runs batched Hugging Face transformer inference while preserving all
    metadata columns (review_text, latitude, longitude, publish_time, user_rating_count).
    """
    if df_reviews.empty or 'review_text' not in df_reviews.columns:
        return df_reviews

    texts = df_reviews['review_text'].tolist()
    predictions = sentiment_analyzer(texts, truncation=True, max_length=512)

    df_reviews['sentiment_label'] = [pred['label'] for pred in predictions]
    df_reviews['sentiment_score'] = [round(pred['score'], 4) for pred in predictions]

    return df_reviews

# --- 5. DASHBOARD INSIGHTS AGGREGATOR ---
def generate_restaurant_insights(df_analyzed: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Aggregates sentiment, extracts dynamic dish/vibe tags, preserves spatial & volume
    metadata for dashboard charts, and ranks the #1 'Rising Star' venue based on model_score.
    """
    if df_analyzed.empty:
        return pd.DataFrame(), {}

    # Cast timestamps to datetime
    if "publish_time" in df_analyzed.columns:
        df_analyzed["publish_time"] = pd.to_datetime(df_analyzed["publish_time"], errors="coerce")

    agg_list = []
    for restaurant, group in df_analyzed.groupby("restaurant_name"):
        total_reviews = len(group)
        pos_reviews = (group["sentiment_label"] == "POSITIVE").sum()
        pos_ratio = round((pos_reviews / total_reviews) * 100, 1)
        avg_rating = round(group["review_rating"].mean(), 1)

        # Preserved metadata fields & PRE-CALCULATED MODEL SCORE
        price_range = group["price_range"].iloc[0]
        cuisine = group["cuisine"].iloc[0]
        lat = group["latitude"].iloc[0] if "latitude" in group.columns else None
        lng = group["longitude"].iloc[0] if "longitude" in group.columns else None
        user_rating_count = group["user_rating_count"].iloc[0] if "user_rating_count" in group.columns else 0

        # Pulling the model score calculated upstream
        model_score = group["model_score"].iloc[0] if "model_score" in group.columns else 0.0

        # Dynamic NLP dish/vibe tags
        pos_texts = group[group["sentiment_label"] == "POSITIVE"]["review_text"].tolist()
        famous_dish, vibe_check = extract_dish_and_vibe(pos_texts if pos_texts else group["review_text"].tolist())

        # Sort chronologically for Recent Top 3 Review card
        if "publish_time" in group.columns and group["publish_time"].notna().any():
            recent_group = group.sort_values(by="publish_time", ascending=False)
        else:
            recent_group = group

        recent_3_reviews = recent_group[
            ["review_text", "publish_time", "review_rating", "sentiment_label"]
        ].head(3).to_dict(orient="records")

        # Top sample quotes
        pos_snippet = group[group["sentiment_label"] == "POSITIVE"]["review_text"].head(1).values
        neg_snippet = group[group["sentiment_label"] == "NEGATIVE"]["review_text"].head(1).values

        agg_list.append({
            "restaurant_name": restaurant,
            "model_score": model_score,  # Passed straight through
            "cuisine": cuisine,
            "price_range": price_range,
            "avg_google_rating": avg_rating,
            "total_google_ratings": user_rating_count,
            "positive_sentiment_pct": pos_ratio,
            "reviews_analyzed": total_reviews,
            "famous_dish": famous_dish,
            "vibe_check": vibe_check,
            "latitude": lat,
            "longitude": lng,
            "recent_3_reviews": recent_3_reviews,
            "sample_positive_review": pos_snippet[0] if len(pos_snippet) > 0 else "N/A",
            "sample_negative_review": neg_snippet[0] if len(neg_snippet) > 0 else "N/A"
        })

    df_scorecard = pd.DataFrame(agg_list)

    # Rank venues strictly by the pre-computed model_score
    df_scorecard = df_scorecard.sort_values(
        by=["model_score", "positive_sentiment_pct"],
        ascending=[False, False]
    ).reset_index(drop=True)

    top_venue = df_scorecard.iloc[0].to_dict() if not df_scorecard.empty else {}

    return df_scorecard, top_venue

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# END-TO-END EXECUTION & OUTPUT TEST
# ==========================================

# 1. COMPUTE SCORES: Run the advanced model math on your existing analyzed data
df_scores = compute_advanced_metrics(df_analyzed)

# Prevent "model_score_x" / "model_score_y" duplication if you run this cell multiple times
if "model_score" in df_analyzed.columns:
    df_analyzed = df_analyzed.drop(columns=["model_score"])

# Attach the new scores to the review-level data
df_analyzed = df_analyzed.merge(
    df_scores[["restaurant_name", "model_score"]],
    on="restaurant_name",
    how="left"
)

# 2. Aggregate insights, extract NLP tags, and build the final scorecard
df_scorecard, top_venue = generate_restaurant_insights(df_analyzed)

# --- PRINT OUTPUTS FOR VERIFICATION ---

print("=== ℹ️ TRANSPARENCY BANNER ===")
try:
    print(df_summary.loc[0, "transparency_note"])
except NameError:
    print("Data loaded from local memory (API filters pre-applied).")

print("\n=== 🏆 RISING STAR VENUE (Ranked by Advanced Model) ===")
print(f"Name:                   {top_venue.get('restaurant_name')}")
print(f"COMPOSITE MODEL SCORE:  {top_venue.get('model_score')} / 100")
print(f"Cuisine:                {top_venue.get('cuisine')}")
print(f"Price Range:            {top_venue.get('price_range')}")
print(f"Positive Sentiment:     {top_venue.get('positive_sentiment_pct')}%")
print(f"Avg Google Rating:      {top_venue.get('avg_google_rating')} ⭐ ({top_venue.get('total_google_ratings')} ratings)")
print(f"Famous For Dish:        {top_venue.get('famous_dish')}")
print(f"Vibe Check:             {top_venue.get('vibe_check')}")
print(f"Coordinates (Lat, Lng): ({top_venue.get('latitude')}, {top_venue.get('longitude')})")

print("\n--- RECENT TOP 3 REVIEWS ---")
for idx, rev in enumerate(top_venue.get("recent_3_reviews", []), 1):
    # Strip line breaks and strictly truncate to ~130 characters (max 2 lines)
    clean_text = str(rev.get('review_text', '')).replace('\n', ' ').strip()
    if len(clean_text) > 130:
        clean_text = clean_text[:127] + "..."

    print(f"{idx}. [{rev['sentiment_label']}] Rating: {rev['review_rating']}⭐ - \"{clean_text}\"")

print("\n=== 📊 FULL RESTAURANT SCORECARD TABLE ===")
display_columns = [
    "restaurant_name",
    "model_score",
    "positive_sentiment_pct",
    "avg_google_rating",
    "price_range",
    "total_google_ratings",
    "famous_dish"
]

# Leaving this without print() so Jupyter renders a clean HTML table
df_scorecard[display_columns].head(10)

=== ℹ️ TRANSPARENCY BANNER ===
Analyzed 20 venues across 100 reviews. 19 using direct menu prices, 1 estimated via local area quartiles.

=== 🏆 RISING STAR VENUE (Ranked by Advanced Model) ===
Name:                   Carluccio’s – Dubai Marina Mall
COMPOSITE MODEL SCORE:  94.92 / 100
Cuisine:                Italian restaurant
Price Range:            AED 50 - 150
Positive Sentiment:     100.0%
Avg Google Rating:      5.0 ⭐ (8070 ratings)
Famous For Dish:        Pizza
Vibe Check:             Italian, Good, Great, Perfect, Special, Attentive, Warm, Friendly, Delicious, Excellent
Coordinates (Lat, Lng): (25.0767664, 55.139438999999996)

--- RECENT TOP 3 REVIEWS ---
1. [POSITIVE] Rating: 5⭐ - "Carluccio’s is one of the best Italian restaurants in town and a place I would gladly recommend. The ambience is warm, elegant,..."
2. [POSITIVE] Rating: 5⭐ - "We had a wonderful dining experience at Carluccio’s Dubai Marina Mall. The food was delicious, the ingredients were fresh, and ..."
3. [POSITI

,restaurant_name,model_score,positive_sentiment_pct,avg_google_rating,price_range,total_google_ratings,famous_dish
0,Carluccio’s – Dubai Marina Mall,94.92,100.0,5.0,AED 50 - 150,8070,Pizza
1,PASTAMAMMA,94.87,100.0,5.0,AED 100 - 300,3466,Pastamama
2,Bussola,93.21,100.0,5.0,AED 150 - 400,2533,Flavor
3,Alloro,93.16,100.0,4.4,AED 50 - 100,5293,Start
4,"Cucina, The Palm",93.02,100.0,5.0,AED 100 - 350,950,Cucina
5,ILOLI - Restaurant in Dubai Marina Walk,92.96,100.0,4.8,AED 50 - 300,1210,Ribeye Steak
6,Eataly at The Beach Dubai,92.72,100.0,4.8,AED 100 - 350,4285,Waiter
7,Ritzi Italian Restaurant Dubai Marina,92.43,100.0,5.0,AED 100 - 250,1530,Dubai
8,Papas Dubai,91.91,100.0,4.8,AED 150 - 450,1697,Detail
9,Villa Verona,90.82,100.0,5.0,AED 100 - 400,683,Villa Verona


In [ ]:
# Save your processed variables to disk
df_scorecard.to_pickle("dashboard_data.pkl")
print("✅ Saved dashboard_data.pkl successfully!")

✅ Saved dashboard_data.pkl successfully!


In [ ]:
%%writefile server.py
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import pandas as pd

app = FastAPI()

# Load dataset
df_scorecard = pd.read_pickle("dashboard_data.pkl")
top_venue = df_scorecard.iloc[0].to_dict() if not df_scorecard.empty else {}

if "price_numeric" not in df_scorecard.columns:
    df_scorecard["price_numeric"] = df_scorecard["price_range"].apply(
        lambda x: int(str(x).split("-")[0].replace("AED", "").strip()) if "-" in str(x) else 150
    )

@app.get("/api/dashboard")
def get_data():
    return {
        "rising_star": top_venue,
        "scorecard": df_scorecard.to_dict(orient="records")
    }

@app.get("/", response_class=HTMLResponse)
def serve_ui():
    return """
    <!DOCTYPE html>
    <html lang="en">
    <head>
      <meta charset="UTF-8">
      <title>Dubai Restaurant Sentiment Dashboard</title>
      <script src="https://unpkg.com/react@18/umd/react.production.min.js"></script>
      <script src="https://unpkg.com/react-dom@18/umd/react-dom.production.min.js"></script>
      <script src="https://unpkg.com/@babel/standalone/babel.min.js"></script>
      <script src="https://cdn.tailwindcss.com"></script>

      <!-- Google Fonts: Plus Jakarta Sans + Outfit -->
      <link rel="preconnect" href="https://fonts.googleapis.com">
      <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
      <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@400;500;600;700;800&family=Plus+Jakarta+Sans:wght@700;800&display=swap" rel="stylesheet">

      <!-- Leaflet Map & Chart.js -->
      <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
      <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
      <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

      <style>
        body {
          font-family: 'Outfit', sans-serif;
          background-color: #f7f4f0;
          color: #2b1820;
        }
        .font-tech-title {
          font-family: 'Plus Jakarta Sans', sans-serif;
          letter-spacing: -0.02em;
        }
        .line-clamp-2 {
          display: -webkit-box;
          -webkit-line-clamp: 2;
          -webkit-box-orient: vertical;
          overflow: hidden;
        }
        .writing-vertical {
          writing-mode: vertical-rl;
          text-orientation: mixed;
        }
      </style>
    </head>
    <body class="p-6 md:p-8 min-h-screen">
      <div id="root"></div>

      <script type="text/babel">
        function Dashboard() {
          const [data, setData] = React.useState(null);
          const [selectedVenue, setSelectedVenue] = React.useState(null);

          const mapRef = React.useRef(null);
          const mapInstance = React.useRef(null);
          const scatterRef = React.useRef(null);
          const scatterInstance = React.useRef(null);
          const barRef = React.useRef(null);
          const barInstance = React.useRef(null);

          React.useEffect(() => {
            fetch('/api/dashboard')
              .then(res => res.json())
              .then(d => {
                setData(d);
                if (d.rising_star && Object.keys(d.rising_star).length > 0) {
                  setSelectedVenue(d.rising_star);
                } else if (d.scorecard && d.scorecard.length > 0) {
                  setSelectedVenue(d.scorecard[0]);
                }
              });
          }, []);

          // Leaflet Map Init
          React.useEffect(() => {
            if (!data || !data.scorecard || !mapRef.current || mapInstance.current) return;
            const firstLat = data.scorecard[0].latitude || 25.077;
            const firstLng = data.scorecard[0].longitude || 55.133;

            const map = L.map(mapRef.current).setView([firstLat, firstLng], 12);
            L.tileLayer('https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png').addTo(map);

            data.scorecard.forEach(r => {
              if (r.latitude && r.longitude) {
                const marker = L.circleMarker([r.latitude, r.longitude], {
                  radius: 8,
                  fillColor: r.positive_sentiment_pct > 85 ? '#721c3b' : '#d98e28',
                  color: '#ffffff',
                  weight: 2,
                  fillOpacity: 0.9
                }).addTo(map);
                marker.bindPopup(`<b>${r.restaurant_name}</b><br>Positive: ${r.positive_sentiment_pct}%`);
                marker.on('click', () => setSelectedVenue(r));
              }
            });
            mapInstance.current = map;
          }, [data]);

          // Scatter Chart Init
          React.useEffect(() => {
            if (!data || !data.scorecard || !scatterRef.current) return;
            if (scatterInstance.current) scatterInstance.current.destroy();

            scatterInstance.current = new Chart(scatterRef.current, {
              type: 'scatter',
              data: {
                datasets: [{
                  label: 'Restaurants',
                  data: data.scorecard.map(r => ({ x: r.price_numeric || 150, y: r.positive_sentiment_pct || 0, name: r.restaurant_name })),
                  backgroundColor: '#721c3b',
                  pointRadius: 7
                }]
              },
              options: {
                responsive: true,
                maintainAspectRatio: false,
                plugins: {
                  legend: { display: false },
                  tooltip: { callbacks: { label: (ctx) => `${ctx.raw.name}: AED ${ctx.raw.x} | ${ctx.raw.y}% Pos` } }
                },
                scales: {
                  x: { grid: { color: '#eadae0' }, ticks: { color: '#59444c', font: { family: 'Outfit' } }, title: { display: true, text: 'Price (AED)', color: '#59444c' } },
                  y: { grid: { color: '#eadae0' }, ticks: { color: '#59444c', font: { family: 'Outfit' } }, title: { display: true, text: 'Positive Sentiment %', color: '#59444c' }, min: 40, max: 100 }
                }
              }
            });
          }, [data]);

          // Bar Chart Init
          React.useEffect(() => {
            if (!data || !data.scorecard || !barRef.current) return;
            if (barInstance.current) barInstance.current.destroy();

            const sorted = [...data.scorecard].sort((a,b) => (b.positive_sentiment_pct||0) - (a.positive_sentiment_pct||0));

            barInstance.current = new Chart(barRef.current, {
              type: 'bar',
              data: {
                labels: sorted.map(r => r.restaurant_name || ''),
                datasets: [{
                  label: 'Positive %',
                  data: sorted.map(r => r.positive_sentiment_pct || 0),
                  backgroundColor: sorted.map(r => (r.positive_sentiment_pct || 0) > 85 ? '#721c3b' : '#d98e28'),
                  borderRadius: 6
                }]
              },
              options: {
                responsive: true,
                maintainAspectRatio: false,
                indexAxis: 'y',
                plugins: { legend: { display: false } },
                scales: {
                  x: { grid: { color: '#eadae0' }, ticks: { color: '#59444c', font: { family: 'Outfit' } }, min: 0, max: 100 },
                  y: { grid: { display: false }, ticks: { color: '#2b1820', font: { family: 'Outfit', weight: '600' } } }
                }
              }
            });
          }, [data]);

          if (!data || !selectedVenue) return <div className="p-12 text-center text-[#721c3b] font-bold">Loading Marketing Analytics...</div>;

          const reviews = selectedVenue.recent_3_reviews || [];
          const rawVibes = String(selectedVenue.vibe_check || "Cozy, Scenic, Romantic, Aesthetic, Authentic, Elegant, Vibrant").split(',').map(v => v.trim());
          const wordCloudStyles = [
            "text-3xl font-tech-title font-black text-[#721c3b] leading-none",
            "text-2xl font-bold text-[#d98e28] leading-tight writing-vertical",
            "text-xl font-extrabold text-[#2b1820]",
            "text-lg font-tech-title font-bold text-[#8c3556]",
            "text-base font-bold text-[#d98e28] leading-none",
            "text-sm font-semibold text-[#59444c] writing-vertical",
            "text-xs font-bold text-[#721c3b]"
          ];

          return (
            <div className="max-w-7xl mx-auto space-y-8">

              {/* PAGE TITLE */}
              <div>
                <h1 className="text-4xl md:text-5xl font-tech-title font-extrabold text-[#721c3b] tracking-tight">
                  Dubai Restaurant Marketing Analytics
                </h1>
                <p className="text-sm font-medium text-[#78616b] mt-1">
                  Sentiment extraction & dynamic guest feedback intelligence
                </p>
              </div>

              {/* 1. TOP PICK HERO SECTION */}
              <div className="bg-white rounded-3xl p-6 md:p-8 border border-[#ebdfd8] shadow-xl shadow-[#721c3b]/5 space-y-6">

                {/* TOP GRID: 3 COLUMNS */}
                <div className="grid grid-cols-1 md:grid-cols-3 gap-6 pb-6 border-b border-[#f0e4dc]">

                  {/* COL 1: TOP PICK BADGE, RESTAURANT NAME, CUISINE */}
                  <div className="space-y-3 flex flex-col justify-center">
                    <div>
                      <span className="inline-flex items-center gap-1.5 bg-[#721c3b]/10 text-[#721c3b] border border-[#721c3b]/20 text-xs font-extrabold px-3.5 py-1.5 rounded-full uppercase tracking-wider">
                        <span>🏆</span> TOP PICK
                      </span>
                    </div>

                    <h2 className="text-4xl md:text-5xl font-tech-title font-extrabold text-[#2b1820] tracking-tight leading-none">
                      {selectedVenue.restaurant_name || "Bussola"}
                    </h2>

                    <p className="text-base font-bold text-[#78616b]">
                      Cuisine: <span className="text-[#721c3b] font-extrabold">{selectedVenue.cuisine || "Italian"}</span>
                    </p>
                  </div>

                  {/* COL 2: GOOGLE RATING & SENTIMENT GAUGE */}
                  <div className="flex flex-col justify-between space-y-4 border-y md:border-y-0 md:border-x border-[#f0e4dc] py-4 md:py-0 md:px-6">
                    <div>
                      <div className="flex items-baseline space-x-2">
                        <span className="text-3xl md:text-4xl font-extrabold text-[#2b1820]">{selectedVenue.avg_google_rating || 5} ★</span>
                        <span className="text-sm font-semibold text-[#78616b]">({selectedVenue.total_google_ratings || 2533} reviews)</span>
                      </div>
                      <p className="text-xs font-bold text-[#721c3b] uppercase tracking-wider mt-1">Google Rating</p>
                    </div>

                    <div>
                      <div className="flex justify-between items-center mb-1.5">
                        <span className="text-xs font-bold text-[#721c3b] uppercase tracking-wider">Sentiment Gauge</span>
                        <span className="text-sm font-extrabold text-[#721c3b]">{selectedVenue.positive_sentiment_pct || 100}% Positive</span>
                      </div>
                      <div className="w-full bg-[#f4ece7] h-3.5 rounded-full overflow-hidden p-0.5 border border-[#eadae0]">
                        <div
                          className="bg-gradient-to-r from-[#d98e28] via-[#8c3556] to-[#721c3b] h-full rounded-full transition-all duration-700 shadow-sm"
                          style={{ width: `${selectedVenue.positive_sentiment_pct || 100}%` }}
                        ></div>
                      </div>
                    </div>
                  </div>

                  {/* COL 3: PRICE RANGE & FAMOUS DISH */}
                  <div className="flex flex-col justify-between space-y-4">
                    <div>
                      <p className="text-2xl md:text-3xl font-extrabold text-[#2b1820] tracking-tight">
                        {selectedVenue.price_range || "50 AED - 150 AED"}
                      </p>
                      <p className="text-xs font-bold text-[#721c3b] uppercase tracking-wider mt-1">Average spend per person</p>
                    </div>

                    <div>
                      <p className="text-2xl md:text-3xl font-tech-title font-bold text-[#721c3b] tracking-tight">
                        {selectedVenue.famous_dish || "Pizza"}
                      </p>
                      <p className="text-xs font-bold text-[#78616b] uppercase tracking-wider mt-1">Famous Dish</p>
                    </div>
                  </div>

                </div>

                {/* BOTTOM GRID: TIGHTENED WORDCLOUD & REVIEWS GAP */}
                <div className="grid grid-cols-1 md:grid-cols-3 gap-6 pt-1 items-start">

                  {/* WORDCLOUD VIBE CHECK (COMPACT SIZE MATCHING REVIEWS) */}
                  <div className="bg-[#fcfaf8] border border-[#eee4dd] rounded-2xl p-3.5 flex flex-col justify-start">
                    <p className="text-xs font-extrabold text-[#721c3b] uppercase tracking-wider mb-2">WordCloud Vibe Check</p>

                    <div className="flex flex-wrap items-center justify-center gap-x-3 gap-y-2 p-1">
                      {rawVibes.map((word, idx) => (
                        <span
                          key={idx}
                          className={`${wordCloudStyles[idx % wordCloudStyles.length]} cursor-default transition-transform hover:scale-105`}
                        >
                          {word}
                        </span>
                      ))}
                    </div>
                  </div>

                  {/* LATEST 3 REVIEWS (REDUCED GAP BELOW HEADER) */}
                  <div className="md:col-span-2">
                    <p className="text-xs font-extrabold text-[#721c3b] uppercase tracking-wider mb-1.5">Latest 3 Reviews</p>

                    <div className="space-y-1.5">
                      {reviews.slice(0, 3).map((rev, i) => (
                        <div key={i} className="bg-[#fcfaf8] border border-[#eee4dd] p-2.5 rounded-xl flex items-start space-x-2.5">
                          <span className="text-[#721c3b] text-xs mt-0.5">🔴</span>
                          <div className="flex-1 min-w-0">
                            <p className="text-xs text-[#3b2730] font-medium leading-snug line-clamp-2">
                              "{String(rev.review_text || '').replace(/\\*/g, '')}"
                            </p>
                          </div>
                          <span className="text-xs font-bold text-[#d98e28] whitespace-nowrap">{rev.review_rating || 5} ★</span>
                        </div>
                      ))}
                    </div>
                  </div>

                </div>

              </div>

              {/* 2. COMPETITOR VISUALIZATIONS GRID */}
              <div className="grid grid-cols-1 md:grid-cols-3 gap-6">
                <div className="bg-white p-5 rounded-2xl border border-[#ebdfd8] shadow-sm">
                  <h3 className="text-xs font-bold text-[#721c3b] uppercase tracking-wider mb-3">Price vs. Sentiment Scatterplot</h3>
                  <div className="h-52 relative"><canvas ref={scatterRef}></canvas></div>
                </div>
                <div className="bg-white p-5 rounded-2xl border border-[#ebdfd8] shadow-sm">
                  <h3 className="text-xs font-bold text-[#721c3b] uppercase tracking-wider mb-3">Positive Sentiment Ranking</h3>
                  <div className="h-52 relative"><canvas ref={barRef}></canvas></div>
                </div>
                <div className="bg-white p-5 rounded-2xl border border-[#ebdfd8] shadow-sm">
                  <h3 className="text-xs font-bold text-[#721c3b] uppercase tracking-wider mb-3">Location Heatmap</h3>
                  <div ref={mapRef} className="h-52 w-full rounded-xl overflow-hidden border border-[#eee4dd]"></div>
                </div>
              </div>

              {/* 3. RESTO SCORECARD TABLE */}
              <div className="bg-white rounded-2xl p-5 border border-[#ebdfd8] shadow-sm">
                <h3 className="text-xs font-bold text-[#721c3b] uppercase tracking-wider mb-4">Resto Scorecard Table (Click row to inspect)</h3>
                <div className="overflow-x-auto">
                  <table className="w-full text-left text-xs text-[#3b2730]">
                    <thead className="bg-[#fcfaf8] text-[#78616b] uppercase font-bold border-b border-[#f0e4dc]">
                      <tr>
                        <th className="p-3">Restaurant</th>
                        <th className="p-3">Price Range</th>
                        <th className="p-3">Rating</th>
                        <th className="p-3">Positive %</th>
                        <th className="p-3">Famous Dish</th>
                        <th className="p-3">Vibe Keywords</th>
                      </tr>
                    </thead>
                    <tbody className="divide-y divide-[#f5ece6]">
                      {data.scorecard.map((r, i) => (
                        <tr
                          key={i}
                          onClick={() => setSelectedVenue(r)}
                          className={`cursor-pointer transition-colors ${selectedVenue.restaurant_name === r.restaurant_name ? 'bg-[#721c3b]/10 font-bold' : 'hover:bg-[#fcfaf8]'}`}
                        >
                          <td className="p-3 font-tech-title font-bold text-base text-[#721c3b]">{r.restaurant_name}</td>
                          <td className="p-3">{r.price_range}</td>
                          <td className="p-3 font-bold text-[#d98e28]">{r.avg_google_rating} ★</td>
                          <td className="p-3 font-extrabold text-[#721c3b]">{r.positive_sentiment_pct}%</td>
                          <td className="p-3 font-medium text-[#2b1820]">{r.famous_dish}</td>
                          <td className="p-3 text-[#78616b]">{r.vibe_check}</td>
                        </tr>
                      ))}
                    </tbody>
                  </table>
                </div>
              </div>

            </div>
          );
        }

        ReactDOM.createRoot(document.getElementById('root')).render(<Dashboard />);
      </script>
    </body>
    </html>
    """

Writing server.py


In [ ]:
import subprocess
import sys
import time
from google.colab import output

if 'server_process' in globals():
    try:
        server_process.terminate()
        server_process.wait(timeout=2)
    except Exception:
        pass

server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(1.5)

print("🔗 Open refreshed Dashboard URL:")
print(output.eval_js("google.colab.kernel.proxyPort(8000)"))

🔗 Open refreshed Dashboard URL:
https://8000-m-s-kkb-usc1b2-5yjbiqdfz4ho-b.us-central1-2.prod.colab.dev
